# FT-CUR Rerun — Tier 1 + Tier 2 (Newton-Schulz pinv)

Re-executa apenas `FTTransformerCURColnorm` nos dois tiers para padronizar
todos os datasets com a pseudo-inversa iterativa (Newton-Schulz, commit `b8f30aa`).

**Motivação:** no Tier 2, os datasets ADULT/BANK/CREDIT/HIGGS50K foram rodados
com SVD, enquanto SHOPPERS/TELCO já usaram a nova pinv. No Tier 1, todos
usaram SVD. Este notebook uniformiza tudo.

**Saída:**
- `ftcur_tier1_rerun.json` — 9 datasets × 30 seeds = 270 runs
- `ftcur_tier2_rerun.json` — 6 datasets × 30 seeds = 180 runs

**Antes de rodar:** Settings → Accelerator → GPU T4 x2 (ou P100).

**Resume:** se a sessão cair, faça download dos JSONs em Output, suba como
dataset Kaggle (Add Data) e ajuste `RESUME_TIER1_PATH` / `RESUME_TIER2_PATH`.

In [ ]:
# ── Célula 1: Verifica GPU ──────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── Célula 2: Clonar repo ───────────────────────────────────────────────────
import os, subprocess

GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'

if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'Dir: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q entmax einops scikit-posthocs openpyxl
import torch, sklearn, numpy
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__}')

In [ ]:
# ── Célula 4: Datasets ──────────────────────────────────────────────────────
# Tier 1: download via script
!python scripts/download_data.py --tier 1

# Tier 2: já presentes como parquet no repo
import sys; sys.path.insert(0, '.')
from src.data.loaders import DatasetLoader

print('\nTier 1:')
for ds in ['BCW', 'PID', 'HAB', 'VCP', 'GCR', 'AUS', 'TWS', 'TWM', 'TWC']:
    X, y, _ = DatasetLoader.load(ds)
    print(f'  {ds:<6}  N={len(y):>5}  p={X.shape[1]}')

print('\nTier 2:')
for ds in ['ADULT', 'BANK', 'CREDIT', 'HIGGS50K', 'SHOPPERS', 'TELCO']:
    X, y, _ = DatasetLoader.load(ds)
    print(f'  {ds:<10}  N={len(y):>6}  p={X.shape[1]}')

In [ ]:
# ── Célula 5: Resume (opcional) ─────────────────────────────────────────────
import shutil, json
from pathlib import Path

OUT_TIER1 = Path('results/ftcur_tier1_rerun.json')
OUT_TIER2 = Path('results/ftcur_tier2_rerun.json')
OUT_TIER1.parent.mkdir(exist_ok=True)

# Se quiser retomar de onde parou:
# 1. Suba os JSONs anteriores como Kaggle dataset (Add Data)
# 2. Ajuste os caminhos abaixo e descomente

# RESUME_TIER1 = Path('/kaggle/input/SEU-DATASET/ftcur_tier1_rerun.json')
# RESUME_TIER2 = Path('/kaggle/input/SEU-DATASET/ftcur_tier2_rerun.json')
# for src, dst in [(RESUME_TIER1, OUT_TIER1), (RESUME_TIER2, OUT_TIER2)]:
#     if src.exists():
#         shutil.copy(src, dst)
#         print(f'Restaurado {dst.name}: {len(json.loads(dst.read_text()))} entries')

for f in [OUT_TIER1, OUT_TIER2]:
    if f.exists():
        n = len(json.loads(f.read_text()))
        print(f'{f.name}: {n} entries já presentes')
    else:
        print(f'{f.name}: começando do zero')

In [ ]:
# ── Célula 6: Grade FT-CUR ──────────────────────────────────────────────────
from src.tuning.grids import grid_size

MODEL = 'FTTransformerCURColnorm'
TIER1_DATASETS = ['BCW', 'PID', 'HAB', 'VCP', 'GCR', 'AUS', 'TWS', 'TWM', 'TWC']
TIER2_DATASETS = ['ADULT', 'BANK', 'CREDIT', 'HIGGS50K', 'SHOPPERS', 'TELCO']
N_SEEDS = 30

g = grid_size(MODEL)
tier1_runs = len(TIER1_DATASETS) * N_SEEDS
tier2_runs = len(TIER2_DATASETS) * N_SEEDS
print(f'Modelo: {MODEL}')
print(f'Grid size: {g}  (5-fold CV → {g*5} fits/run)')
print(f'Tier 1: {tier1_runs} runs  ({len(TIER1_DATASETS)} datasets × {N_SEEDS} seeds)')
print(f'Tier 2: {tier2_runs} runs  ({len(TIER2_DATASETS)} datasets × {N_SEEDS} seeds)')
print(f'Total: {tier1_runs + tier2_runs} runs')

In [ ]:
# ── Célula 7: Rodar Tier 1 ──────────────────────────────────────────────────
datasets_str = ' '.join(TIER1_DATASETS)
seeds_str    = ' '.join(map(str, range(N_SEEDS)))

!python -u scripts/run_tier1_gridcv.py \
    --models {MODEL} \
    --datasets {datasets_str} \
    --output results/ftcur_tier1_rerun.json \
    2>&1 | tee /kaggle/working/run_tier1.log

# Cópia imediata para /kaggle/working (caso a sessão caia antes da célula 9)
import shutil
shutil.copy('results/ftcur_tier1_rerun.json', '/kaggle/working/ftcur_tier1_rerun.json')
print('\nTier 1 concluído — arquivo salvo em /kaggle/working/')

In [ ]:
# ── Célula 8: Rodar Tier 2 ──────────────────────────────────────────────────
datasets_str = ' '.join(TIER2_DATASETS)
seeds_str    = ' '.join(map(str, range(N_SEEDS)))

!python -u scripts/run_tier2_gridcv.py \
    --models {MODEL} \
    --datasets {datasets_str} \
    --seeds {seeds_str} \
    --n-train 2000 \
    --output results/ftcur_tier2_rerun.json \
    2>&1 | tee /kaggle/working/run_tier2.log

shutil.copy('results/ftcur_tier2_rerun.json', '/kaggle/working/ftcur_tier2_rerun.json')
print('\nTier 2 concluído — arquivo salvo em /kaggle/working/')

In [ ]:
# ── Célula 9: Resumo final ──────────────────────────────────────────────────
import json, statistics as st
from pathlib import Path
from collections import defaultdict

for label, path, n_expected in [
    ('Tier 1', 'results/ftcur_tier1_rerun.json', len(TIER1_DATASETS) * N_SEEDS),
    ('Tier 2', 'results/ftcur_tier2_rerun.json', len(TIER2_DATASETS) * N_SEEDS),
]:
    records = json.loads(Path(path).read_text())
    ok = [r for r in records if r.get('status') == 'ok']
    print(f'\n=== {label} ===')
    print(f'Completos: {len(ok)}/{n_expected} ({len(ok)/n_expected*100:.0f}%)')

    f1_by_ds = defaultdict(list)
    for r in ok:
        f1_by_ds[r['dataset']].append(r['test_f1_macro'])

    print(f'{"Dataset":<12}  F1-macro  n')
    overall = []
    for ds, vals in sorted(f1_by_ds.items()):
        print(f'  {ds:<10}  {st.mean(vals):.4f} ± {st.stdev(vals):.4f}  {len(vals)}')
        overall.extend(vals)
    if overall:
        print(f'  {"MÉDIA":<10}  {st.mean(overall):.4f}')

# Copiar ambos para output raiz
import shutil
shutil.copy('results/ftcur_tier1_rerun.json', '/kaggle/working/ftcur_tier1_rerun.json')
shutil.copy('results/ftcur_tier2_rerun.json', '/kaggle/working/ftcur_tier2_rerun.json')
print('\nArquivos disponíveis em Output para download.')